# Playground

## Imports

In [25]:
import sys
from pathlib import Path
from fractions import Fraction

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from symbolica import Expression, S

from model import (
    COLOR_ADJ_INDEX,
    COLOR_FUND_INDEX,
    LORENTZ_INDEX,
    SPINOR_INDEX,
    WEAK_ADJ_INDEX,
    WEAK_FUND_INDEX,
    ComplexScalarKineticTerm,
    CovD,
    DiracKineticTerm,
    Field,
    FieldStrength,
    Gamma,
    GaugeFixing,
    GaugeGroup,
    GaugeKineticTerm,
    GaugeRepresentation,
    GhostLagrangian,
    Model,
)
from symbolic.spenso_structures import (
    gauge_generator,
    structure_constant,
    weak_gauge_generator,
    weak_structure_constant,
)
from symbolic.vertex_engine import I


def expr_text(expr):
    text = expr.to_canonical_string() if hasattr(expr, "to_canonical_string") else str(expr)
    return text.replace("spenso::", "")


def signature_rows(signatures):
    return tuple(
        {
            "fields": ", ".join(signature.names),
            "arity": signature.arity,
            "terms": signature.term_count,
            "sectors": ", ".join(signature.sectors),
        }
        for signature in signatures
    )


def report_row(report):
    return {
        "matched_signatures": report.matched_signatures,
        "matched_terms": report.matched_terms,
        "total_signatures": report.total_signatures,
        "total_terms": report.total_terms,
        "signatures": signature_rows(report.signatures),
    }


def validation_rows(report):
    return {
        "ok": report.ok,
        "issues": tuple(
            {
                "code": issue.code,
                "severity": issue.severity,
                "message": issue.message,
            }
            for issue in report.issues
        ),
    }


## Symbols

In [26]:
mu, nu, rho = S("mu", "nu", "rho")
eQED, gS, g2, xiQCD = S("eQED", "gS", "g2", "xiQCD")
qPhi, qPsi, qQ = S("qPhi", "qPsi", "qQ")
lam, y, g4, gGamma = S("lam", "y", "g4", "gGamma")


## Fields

In [27]:
Phi = Field(
    "Phi",
    spin=0,
    self_conjugate=False,
    symbol=S("phi"),
    conjugate_symbol=S("phidag"),
    quantum_numbers={"Q": qPhi},
)
Chi = Field(
    "Chi",
    spin=0,
    self_conjugate=False,
    symbol=S("chi"),
    conjugate_symbol=S("chidag"),
)
Psi = Field(
    "Psi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psi"),
    conjugate_symbol=S("psibar"),
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Q": qPsi},
)
Xi = Field(
    "Xi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("xi"),
    conjugate_symbol=S("xibar"),
    indices=(SPINOR_INDEX,),
)
Quark = Field(
    "q",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("q"),
    conjugate_symbol=S("qbar"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
    quantum_numbers={"Q": qQ},
)
Photon = Field("A", spin=1, self_conjugate=True, symbol=S("A0"), indices=(LORENTZ_INDEX,))
Gluon = Field("G", spin=1, self_conjugate=True, symbol=S("G0"), indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX))
GhostG = Field(
    "ghG",
    spin=0,
    kind="ghost",
    self_conjugate=False,
    symbol=S("ghG0"),
    conjugate_symbol=S("ghGbar0"),
    indices=(COLOR_ADJ_INDEX,),
)
W = Field("W", spin=1, self_conjugate=True, symbol=S("W0"), indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX))
GhostW = Field(
    "ghW",
    spin=0,
    kind="ghost",
    self_conjugate=False,
    symbol=S("ghW0"),
    conjugate_symbol=S("ghWbar0"),
    indices=(WEAK_ADJ_INDEX,),
)
H = Field(
    "H",
    spin=0,
    self_conjugate=False,
    symbol=S("H0"),
    conjugate_symbol=S("Hdag0"),
    indices=(WEAK_FUND_INDEX,),
)
L = Field(
    "L",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("L0"),
    conjugate_symbol=S("Lbar0"),
    indices=(SPINOR_INDEX, WEAK_FUND_INDEX),
)

(Phi, Chi, Psi, Xi, Quark, Photon, Gluon, GhostG, W, GhostW, H, L)


(Field(name='Phi', spin=0, self_conjugate=False, indices=(), kind='scalar', statistics='boson', symbol=phi, conjugate_symbol=phidag, mass=None, quantum_numbers={'Q': qPhi}),
 Field(name='Chi', spin=0, self_conjugate=False, indices=(), kind='scalar', statistics='boson', symbol=chi, conjugate_symbol=chidag, mass=None, quantum_numbers={}),
 Field(name='Psi', spin=Fraction(1, 2), self_conjugate=False, indices=(IndexType(name='Spinor', representation=Representation { rep: SelfDual(1), dim: Concrete(4) }, kind='spinor', prefix='i'),), kind='fermion', statistics='fermion', symbol=psi, conjugate_symbol=psibar, mass=None, quantum_numbers={'Q': qPsi}),
 Field(name='Xi', spin=Fraction(1, 2), self_conjugate=False, indices=(IndexType(name='Spinor', representation=Representation { rep: SelfDual(1), dim: Concrete(4) }, kind='spinor', prefix='i'),), kind='fermion', statistics='fermion', symbol=xi, conjugate_symbol=xibar, mass=None, quantum_numbers={}),
 Field(name='q', spin=Fraction(1, 2), self_conjug

## Gauge Representations

In [28]:
COLOR_FUND_REP = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=gauge_generator,
    name="fund",
)
WEAK_DOUBLET_REP = GaugeRepresentation(
    index=WEAK_FUND_INDEX,
    generator_builder=weak_gauge_generator,
    name="doublet",
)

(COLOR_FUND_REP, WEAK_DOUBLET_REP)


(GaugeRepresentation(index=IndexType(name='ColorFund', representation=Representation { rep: Dualizable(3), dim: Concrete(3) }, kind='color_fund', prefix='c'), generator_builder=<function gauge_generator at 0x10aa9c720>, name='fund', slot=None, slot_policy='unique'),
 GaugeRepresentation(index=IndexType(name='WeakFund', representation=Representation { rep: Dualizable(3), dim: Concrete(2) }, kind='weak_fund', prefix='w'), generator_builder=<function weak_gauge_generator at 0x10aa9c880>, name='doublet', slot=None, slot_policy='unique'))

## Gauge Groups

In [29]:
U1QED = GaugeGroup(
    name="U1QED",
    abelian=True,
    coupling=eQED,
    gauge_boson=Photon.symbol,
    charge="Q",
)
SU3C = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=gS,
    gauge_boson=Gluon.symbol,
    ghost_field=GhostG.symbol,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)
SU2L = GaugeGroup(
    name="SU2L",
    abelian=False,
    coupling=g2,
    gauge_boson=W.symbol,
    ghost_field=GhostW.symbol,
    structure_constant=weak_structure_constant,
    representations=(WEAK_DOUBLET_REP,),
)

(U1QED, SU3C, SU2L)


(GaugeGroup(name='U1QED', abelian=True, coupling=eQED, gauge_boson=A0, ghost_field=None, structure_constant=None, representations=(), charge='Q'),
 GaugeGroup(name='SU3C', abelian=False, coupling=gS, gauge_boson=G0, ghost_field=ghG0, structure_constant=<function structure_constant at 0x10aa9c7d0>, representations=(GaugeRepresentation(index=IndexType(name='ColorFund', representation=Representation { rep: Dualizable(3), dim: Concrete(3) }, kind='color_fund', prefix='c'), generator_builder=<function gauge_generator at 0x10aa9c720>, name='fund', slot=None, slot_policy='unique'),), charge=None),
 GaugeGroup(name='SU2L', abelian=False, coupling=g2, gauge_boson=W0, ghost_field=ghW0, structure_constant=<function weak_structure_constant at 0x10aa9c930>, representations=(GaugeRepresentation(index=IndexType(name='WeakFund', representation=Representation { rep: Dualizable(3), dim: Concrete(2) }, kind='weak_fund', prefix='w'), generator_builder=<function weak_gauge_generator at 0x10aa9c880>, name='

## Local Models

In [30]:
local_yukawa_model = Model(fields=(Psi, Phi), lagrangian_decl=y * Psi.bar * Psi * Phi)
local_four_fermion_model = Model(fields=(Psi, Xi), lagrangian_decl=g4 * Psi.bar * Psi * Xi.bar * Xi)
local_quartic_model = Model(fields=(Phi, Chi), lagrangian_decl=lam * Phi.bar * Phi * Chi.bar * Chi)
gamma_model = Model(
    fields=(Psi,),
    lagrangian_decl=(
        gGamma * Psi.bar * Gamma(mu) * Gamma(nu) * Psi
        + gGamma * Psi.bar * Gamma(nu) * Gamma(mu) * Psi
    ),
)

## Basic Model

In [31]:
basic_model = Model(
    gauge_groups=(U1QED, SU3C),
    fields=(Phi, Psi, Quark, Photon, Gluon, GhostG),
    lagrangian_decl=(
        CovD(Phi.bar, mu) * CovD(Phi, mu)
        + I * Psi.bar * Gamma(mu) * CovD(Psi, mu)
        + I * Quark.bar * Gamma(mu) * CovD(Quark, mu)
        + (-(Expression.num(1) / Expression.num(4)) * FieldStrength(U1QED, mu, nu) * FieldStrength(U1QED, mu, nu))
        + (-(Expression.num(1) / Expression.num(4)) * FieldStrength(SU3C, mu, nu) * FieldStrength(SU3C, mu, nu))
        + GaugeFixing(SU3C, xi=xiQCD)
        + GhostLagrangian(SU3C)
    ),
)
compiled = basic_model.lagrangian()

{
    "term_count": len(compiled.terms),
    "signature_count": len(compiled.vertex_signatures()),
}


{('Phi.bar', 'Phi', 'A'): -1𝑖*eQED*qPhi*(2*𝜋)^d*pcomp(q1,mu)*Delta(q1+q2+q3)+1𝑖*eQED*qPhi*(2*𝜋)^d*pcomp(q2,mu)*Delta(q1+q2+q3), ('Phi.bar', 'Phi', 'A', 'A'): 2𝑖*eQED^2*qPhi^2*(2*𝜋)^d*spenso::g(spenso::mink(4,mu3),spenso::mink(4,mu4))*Delta(q1+q2+q3+q4), ('Phi.bar', 'Phi'): -1𝑖*(2*𝜋)^d*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)*Delta(q1+q2), ('Psi.bar', 'Psi', 'A'): -1𝑖*eQED*qPsi*(2*𝜋)^d*spenso::gamma(spenso::bis(4,i1),spenso::bis(4,i2),spenso::mink(4,mu3))*Delta(q1+q2+q3), ('Psi.bar', 'Psi'): 1𝑖*(2*𝜋)^d*spenso::gamma(spenso::bis(4,i1),spenso::bis(4,i2),spenso::mink(4,mu1_int))*pcomp(q2,mu1_int)*Delta(q1+q2), ('q.bar', 'q', 'A'): -1𝑖*eQED*qQ*(2*𝜋)^d*spenso::g(spenso::cof(3,c1),spenso::cof(3,c2))*spenso::gamma(spenso::bis(4,i1),spenso::bis(4,i2),spenso::mink(4,mu3))*Delta(q1+q2+q3), ('q.bar', 'q', 'G'): -1𝑖*gS*(2*𝜋)^d*spenso::gamma(spenso::bis(4,i1),spenso::bis(4,i2),spenso::mink(4,mu3))*spenso::t(spenso::coad(8,a3),spenso::cof(3,c1),spenso::cof(3,c2))*Delta(q1+q2+q3), ('q.bar', 'q'): 1𝑖*(2*𝜋)^

## Vertex Output

In [32]:
{
    "qbar_q_A": expr_text(compiled.feynman_rule(Quark.bar, Quark, Photon)),
    "qbar_q_G": expr_text(compiled.feynman_rule(Quark.bar, Quark, Gluon)),
    "psi_psi_A_no_delta": expr_text(compiled.feynman_rule(Psi.bar, Psi, Photon, include_delta=False)),
    "yukawa_unstripped": expr_text(local_yukawa_model.lagrangian().feynman_rule(Psi.bar, Psi, Phi, simplify=False, strip_externals=False)),
    "gamma_simplified": expr_text(gamma_model.lagrangian().feynman_rule(Psi.bar, Psi, simplify=True, simplify_gamma=True)),
}


qbar q A vertex:  -1𝑖*eQED*qQ*(2*𝜋)^d*g(cof(3, c1),cof(3, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*Delta(q1+q2+q3)
qbar q G vertex:  -1𝑖*gS*(2*𝜋)^d*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))*Delta(q1+q2+q3)
psi psi A vertex (no delta):  -1𝑖*eQED*qPsi*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))
yukawa unstripped:  1𝑖*y*(2*𝜋)^d*g(bis(4, i1),bis(4, i2))*delta(phi,phi)*delta(psi,psi)*delta(psibar,psibar)*U(phi,q3)*Delta(q1+q2+q3)
gamma simplified:  2𝑖*gGamma*(2*𝜋)^d*g(mink(4, mu),mink(4, nu))*g(bis(4, i1),bis(4, i2))*Delta(q1+q2)


## Vertex Reports

In [33]:
{
    "arity_3": signature_rows(compiled.vertex_signatures(arity=3)),
    "exact_qbar_q_G": signature_rows(compiled.vertex_signatures(signature=(Quark.bar, Quark, Gluon))),
    "contains_ghosts": signature_rows(compiled.vertex_signatures(contains_fields=(GhostG.bar, GhostG))),
    "report_qbar_q_G": report_row(compiled.vertex_report(signature=(Quark.bar, Quark, Gluon))),
}


all: (VertexSignature(fields=(Field(name='A', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', prefix='mu'),), kind='vector', statistics='boson', symbol=A0, conjugate_symbol=None, mass=None, quantum_numbers={}), Field(name='A', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', prefix='mu'),), kind='vector', statistics='boson', symbol=A0, conjugate_symbol=None, mass=None, quantum_numbers={})), names=('A', 'A'), arity=2, term_count=2, sectors=('pure_gauge',)), VertexSignature(fields=(Field(name='G', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', prefix='mu'), IndexType(name='ColorAdj', representation=Representation { rep: SelfDual(2), dim: Concrete(8) }, kind='color_adj', prefi

## Model Validation

In [34]:
bad_ghost_model = Model(
    gauge_groups=(U1QED,),
    fields=(Photon,),
    lagrangian_decl=GhostLagrangian(U1QED),
)
bad_rep_model = Model(gauge_groups=(SU3C,), fields=(Phi, Gluon))
bad_rep_model.covariant_terms = (ComplexScalarKineticTerm(field=Phi, gauge_group=SU3C),)
bad_kinetic_model = Model(
    fields=(Phi,),
    lagrangian_decl=ComplexScalarKineticTerm(field=Phi, coefficient=2),
)

{
    "basic": validation_rows(basic_model.validate()),
    "bad_ghost": validation_rows(bad_ghost_model.validate()),
    "bad_rep": validation_rows(bad_rep_model.validate()),
    "bad_kinetic": validation_rows(bad_kinetic_model.validate()),
}


{'basic': ValidationReport(issues=()),
 'bad_ghost': ValidationReport(issues=(ValidationIssue(code='abelian_ghost_sector', message="Ghost validation only supports non-abelian gauge groups; got 'U1QED'.", severity='error'),)),
 'bad_rep': ValidationReport(issues=(ValidationIssue(code='missing_gauge_representation', message="Covariant validation requires field 'Phi' to carry a declared representation under gauge group 'SU3C'. Field indices: (none). Declared representation indices: ColorFund.", severity='error'),)),
 'bad_kinetic': ValidationReport(issues=(ValidationIssue(code='kinetic_normalization', message="complex-scalar kinetic term for field 'Phi' has non-canonical coefficient 2; expected 1.", severity='error'),))}

## Compiled Validation

In [35]:
same_name_1 = Field("Phi", spin=0, self_conjugate=False, symbol=S("phi1"), conjugate_symbol=S("phi1dag"))
same_name_2 = Field("Phi", spin=0, self_conjugate=False, symbol=S("phi2"), conjugate_symbol=S("phi2dag"))
mass_mixing_model = Model(fields=(Phi, Chi), lagrangian_decl=S("m12") * Phi.bar * Chi)
same_name_mixing_model = Model(fields=(same_name_1, same_name_2), lagrangian_decl=S("mSame") * same_name_1.bar * same_name_2)
noncanonical_pair_model = Model(fields=(Phi, Chi), lagrangian_decl=S("cNon") * Phi * Chi)

{
    "mass_mixing": validation_rows(mass_mixing_model.lagrangian().validate()),
    "same_name_mixing": validation_rows(same_name_mixing_model.lagrangian().validate()),
    "noncanonical_pair": validation_rows(noncanonical_pair_model.lagrangian().validate()),
}


{'mass_mixing': ValidationReport(issues=(ValidationIssue(code='mass_structure_mixing', message="Off-diagonal scalar mass-like bilinear detected between fields 'Phi' and 'Chi'; compiled term has 0 derivatives and only 2 matter fields.", severity='warning'),)),
 'same_name_mixing': ValidationReport(issues=(ValidationIssue(code='mass_structure_mixing', message="Off-diagonal scalar mass-like bilinear detected between fields 'Phi' and 'Phi'; compiled term has 0 derivatives and only 2 matter fields.", severity='warning'),)),
 'noncanonical_pair': ValidationReport(issues=())}

## Sector Counts

In [36]:
all_gg = [sig for sig in compiled.vertex_signatures() if sig.names == ("G", "G")]
pure_gg = [sig for sig in compiled.vertex_signatures(sector="pure_gauge") if sig.names == ("G", "G")]
gauge_fixing_gg = [sig for sig in compiled.vertex_signatures(sector="gauge_fixing") if sig.names == ("G", "G")]

{
    "GG_all": signature_rows(all_gg),
    "GG_pure_gauge": signature_rows(pure_gg),
    "GG_gauge_fixing": signature_rows(gauge_fixing_gg),
    "pure_gauge": report_row(compiled.vertex_report(sector="pure_gauge")),
    "gauge_fixing": report_row(compiled.vertex_report(sector="gauge_fixing")),
    "ghost": report_row(compiled.vertex_report(sector="ghost")),
}


{'all': VertexSignature(fields=(Field(name='G', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', prefix='mu'), IndexType(name='ColorAdj', representation=Representation { rep: SelfDual(2), dim: Concrete(8) }, kind='color_adj', prefix='a')), kind='vector', statistics='boson', symbol=G0, conjugate_symbol=None, mass=None, quantum_numbers={}), Field(name='G', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', prefix='mu'), IndexType(name='ColorAdj', representation=Representation { rep: SelfDual(2), dim: Concrete(8) }, kind='color_adj', prefix='a')), kind='vector', statistics='boson', symbol=G0, conjugate_symbol=None, mass=None, quantum_numbers={})), names=('G', 'G'), arity=2, term_count=3, sectors=('gauge_fixing', 'pure_gauge')),
 'pure_gauge': VertexSignature(fields=(Field(name='G', sp